# Week 22, Day 4: Excitability: the FitzHugh-Nagumo neuron

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w022/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Find the resting state

Write `rest(I)`: substitute the w-nullcline into the v-nullcline to get a cubic in v, find its real root with np.roots, and return (v, w). Then write the analytic Jacobian `jac(x)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
a, b, eps = 0.7, 0.8, 0.08
def fhn(x, I=0.0):
    v, w = x
    return np.array([v - v ** 3 / 3 - w + I, eps * (v + a - b * w)])

def rest(I):
    # v - v^3/3 - (v + a)/b + I = 0, written as a cubic in v
    roots = np.roots([-1 / 3, 0, ..., ...])
    v = roots[np.abs(roots.imag) < 1e-9].real[0]
    return np.array([v, (v + a) / b])

def jac(x):
    v, w = x
    return np.array([[..., -1.0], [eps, -eps * b]])

x0 = rest(0.0); J = jac(x0)
print("rest:", x0.round(4), " trace", round(np.trace(J), 4), " det", round(np.linalg.det(J), 4), " eigenvalues", np.linalg.eigvals(J).round(4))

In [ ]:
import numpy as np
def test_rest():
    "the resting state at I = 0 is near v = -1.199, w = -0.624, and both derivatives vanish there"
    x = rest(0.0)
    assert abs(x[0] + 1.1994) < 1e-3 and abs(x[1] + 0.6243) < 1e-3 and np.abs(fhn(x, 0.0)).max() < 1e-9

def test_jac():
    "the Jacobian at rest has trace about -0.503 and the state is a stable spiral"
    J = jac(rest(0.0))
    assert abs(np.trace(J) + 0.5026) < 1e-3 and np.all(np.linalg.eigvals(J).real < 0) and np.any(np.abs(np.linalg.eigvals(J).imag) > 0)

test_rest()
test_jac()
print("All checks passed")

### Exercise: All or none

Write `peak(dv)`, which kicks v by dv from rest and returns the largest v in the next 100 time units (RK4, dt = 0.05). Compute `peaks` for the kicks given, then bisect between 0.4 and 0.8 (30 halvings, using peak above 1 as the test) to find `kick_th`.

In [ ]:
def simulate(x0, I=0.0, T=100.0, dt=0.05):
    x = np.array(x0, float); vs = np.empty(int(T / dt))
    for k in range(len(vs)):
        k1 = fhn(x, I); k2 = fhn(x + dt / 2 * k1, I); k3 = fhn(x + dt / 2 * k2, I); k4 = fhn(x + dt * k3, I)
        x = x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4); vs[k] = x[0]
    return vs

def peak(dv):
    return ...

kicks = np.round(np.arange(0.1, 1.51, 0.1), 2)
peaks = np.array([peak(d) for d in kicks])
lo, hi = 0.4, 0.8
for _ in range(30):
    mid = (lo + hi) / 2
    if ...: hi = mid
    else: lo = mid
kick_th = (lo + hi) / 2
print("peaks:", dict(zip(kicks.tolist(), peaks.round(2).tolist())))
print(f"threshold kick {kick_th:.4f}")
plt.plot(kicks, peaks, "o-"); plt.xlabel("kick in v"); plt.ylabel("peak v"); plt.show()

In [ ]:
import numpy as np
def test_jump():
    "peak v is below 0 for a kick of 0.4 and above 1.5 for a kick of 0.8"
    assert peaks[list(kicks).index(0.4)] < 0 and peaks[list(kicks).index(0.8)] > 1.5

def test_threshold():
    "the bisected threshold kick lies between 0.5 and 0.6"
    assert 0.5 < kick_th < 0.6

test_jump()
test_threshold()
print("All checks passed")

### Exercise: Where rest gives way

The trace vanishes where 1 − v² = εb. Take the root on the left side, v_h = −√(1 − εb), and compute the input `I_hopf` that puts the resting state there (from the v-nullcline, using w = (v + a)/b). Then simulate 400 time units at I = 0.25 and 0.4 from just beside rest and store the range (max minus min) of v over the last 200 in `swing[I]`.

In [ ]:
v_h = -np.sqrt(1 - eps * b)
I_hopf = ...
swing = {}
for I in (0.25, 0.4):
    vs = simulate(rest(I) + np.array([0.01, 0.0]), I, T=400.0)
    swing[I] = ...
print(f"I_hopf = {I_hopf:.4f};  swing of v: {({k: round(float(v), 4) for k, v in swing.items()})}")

In [ ]:
import numpy as np
def test_ihopf():
    "I_hopf is about 0.3313"
    assert abs(I_hopf - 0.3313) < 1e-3

def test_trace():
    "the trace of the Jacobian at rest is negative just below I_hopf and positive just above"
    assert np.trace(jac(rest(I_hopf - 0.01))) < 0 < np.trace(jac(rest(I_hopf + 0.01)))

def test_osc():
    "the late swing of v is below 0.01 at I = 0.25 and above 3 at I = 0.4"
    assert swing[0.25] < 0.01 and swing[0.4] > 3

test_ihopf()
test_trace()
test_osc()
print("All checks passed")